In [9]:
!curl -L -o ~/Downloads/student-demographics-online-education-dataoulad.zip \
  "https://www.kaggle.com/api/v1/datasets/download/anlgrbz/student-demographics-online-education-dataoulad"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0Warning: Failed to create the file 
  0 42.1M    0  1177    0     0   1530      0  8:01:27 --:--:--  8:01:27  5658
curl: (23) Failure writing output to destination


In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import xgboost as xgb

In [12]:
pip install kagglehub pandas

# 1. Download latest version of the dataset from Kaggle automatically

In [19]:
import os
import pandas as pd
import kagglehub

# 1. Download/get the dataset path
path = kagglehub.dataset_download("anlgrbz/student-demographics-online-education-dataoulad")
print("Dataset path:", path)

# 2. List files in the downloaded directory to find the exact filename
files = os.listdir(path)
print("Files available:", files)

# 3. Dynamically find the student info CSV file
csv_filename = next((f for f in files if 'studentInfo' in f.lower() or 'student_info' in f.lower()), files[0])
csv_file_path = os.path.join(path, csv_filename)

print(f"Loading file from: {csv_file_path}")

# 4. Read with pandas
df = pd.read_csv(csv_file_path)
print("Successfully loaded DataFrame! Shape:", df.shape)
print(df.head())

Dataset path: /root/.cache/kagglehub/datasets/anlgrbz/student-demographics-online-education-dataoulad/versions/1
Files available: ['courses.csv', 'studentAssessment.csv', 'studentRegistration.csv', 'vle.csv', 'assessments.csv', 'studentInfo.csv', 'studentVle.csv']
Loading file from: /root/.cache/kagglehub/datasets/anlgrbz/student-demographics-online-education-dataoulad/versions/1/courses.csv
Successfully loaded DataFrame! Shape: (22, 3)
  code_module code_presentation  module_presentation_length
0         AAA             2013J                         268
1         AAA             2014J                         269
2         BBB             2013J                         268
3         BBB             2014J                         262
4         BBB             2013B                         240


In [20]:
# 1. Download dataset automatically via kagglehub & load dynamically
path = kagglehub.dataset_download("anlgrbz/student-demographics-online-education-dataoulad")
print("Dataset path:", path)

files = os.listdir(path)
csv_filename = next((f for f in files if 'studentinfo' in f.lower()), files[0])
csv_file_path = os.path.join(path, csv_filename)

print(f"Loading file from: {csv_file_path}")
df = pd.read_csv(csv_file_path)

Dataset path: /root/.cache/kagglehub/datasets/anlgrbz/student-demographics-online-education-dataoulad/versions/1
Loading file from: /root/.cache/kagglehub/datasets/anlgrbz/student-demographics-online-education-dataoulad/versions/1/studentInfo.csv


In [21]:
# 2. Preprocessing & Feature Engineering
features = ['num_of_prev_attempts', 'studied_credits']
X = df[features].fillna(0)
y = df['final_result'].apply(lambda val: 1 if val in ['Pass', 'Distinction'] else 0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [22]:
# 3. Dimensionality Reduction & Unsupervised Clustering (PCA & K-Means)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

In [23]:
# 4. Supervised Training Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
import xgboost as xgb

# Streamlined algorithm registry (faster execution)
models = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(n_estimators=50), # reduced estimators for speed
    "Naive Bayes": GaussianNB(),
    "k-Nearest Neighbors": KNeighborsClassifier(),
    "XGBoost": xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')
}

results = {}
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    results[name] = model.score(X_test, y_test)

print("\nModel Performance Summary:")
for model_name, score in results.items():
    print(f"- {model_name}: {score:.4f}")

Training Logistic Regression...
Training Random Forest...
Training Naive Bayes...
Training k-Nearest Neighbors...
Training XGBoost...


/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [14:06:36] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Model Performance Summary:
- Logistic Regression: 0.5633
- Random Forest: 0.5628
- Naive Bayes: 0.5559
- k-Nearest Neighbors: 0.5599
- XGBoost: 0.5642


In [26]:
# 6. Export processed features/cluster metrics for Django consumption
os.makedirs('equitable_education/data', exist_ok=True)
id_col = 'id_student' if 'id_student' in df.columns else df.columns[0]

output_data = df[[id_col, 'code_module', 'final_result']].copy()
output_data['cluster'] = cluster_labels
output_data.to_json('equitable_education/data/processed_telemetry.json', orient='records')
print("\nSuccessfully exported telemetry metrics to 'equitable_education/data/processed_telemetry.json'!")


Successfully exported telemetry metrics to 'equitable_education/data/processed_telemetry.json'!
